# Appendix. 01 - HTS preprocessing and expansion

This notebook prepares the **weighted HTS trip table** used in the OD-level comparison.
It follows the manuscript flow:

1. Load raw HTS person records  
2. Preprocess person attributes  
3. Derive person-level expansion coefficients  
4. Merge person weights into trip records  
5. Retain intra-Seoul trips only  
6. Convert the weighted HTS trips to a common structure aligned with LCS

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# Base directories
BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "0_data"
POP_DIR = BASE_DIR / "0_data"
PREP_DIR = BASE_DIR / "1_data_preprocessing"
PREP_DIR.mkdir(parents=True, exist_ok=True)

RESULT_DIR = BASE_DIR / "2_data_OD"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

Additional_RESULT_DIR = BASE_DIR / "9_Result"
Additional_RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Raw HTS files
HTS_PERSON_FILE = RAW_DIR / "개인통행실태조사(2021년기준)" / "①개인특성.csv"
HTS_TRIP_FILE = RAW_DIR / "개인통행실태조사(2021년기준)" / "②이동특성.csv"

# Auxiliary files
ZONE_ID_FILE = RAW_DIR / "administrative districts (2016-2021).xlsx"
POP_KOREA_FILE = POP_DIR / "202110_연령별인구현황.xlsx"
POP_SEOUL_FILE = POP_DIR / "202110_연령별인구현황_서울시.xlsx"

# Output files
HTS_PERSON_WEIGHTED_FILE = PREP_DIR / "HTS_person_weighted.csv"
HTS_TRIP_WEIGHTED_FILE = PREP_DIR / "HTS_trip_weighted.csv"

OD_HTS_FILE = RESULT_DIR / "OD_HTS.csv"
OD_HTS_GENDER_FILE = RESULT_DIR / "OD_HTS_gender.csv"
OD_HTS_AGEGROUP_FILE = RESULT_DIR / "OD_HTS_agegroup.csv"
OD_HTS_TRAVELTYPE_FILE = RESULT_DIR / "OD_HTS_traveltype.csv"
OD_HTS_TIMEZONE_FILE = RESULT_DIR / "OD_HTS_timezone.csv"
OD_HTS_TRAVELTYPE_TIMEZONE_FILE = RESULT_DIR / "OD_HTS_traveltype_timezone.csv"

AGE_GROUPS = [
    "0-9 years",
    "10-19 years",
    "20-29 years",
    "30-39 years",
    "40-49 years",
    "50-59 years",
    "60-69 years",
    "70-79 years",
    "80 years and above",
]

SEX_MAP = {1: "Male", 2: "Female"}

SIDO_MAP = {
    "서울": "서울특별시", # Seoul
    "인천": "인천광역시", # Incheon
    "경기": "경기도",     # Gyeonggi
    "강원": "강원도",     # Gangwon
    "경남": "경상남도",   # Gyeongnam
    "경북": "경상북도",   # Gyeongbuk
    "광주": "광주광역시", # Gwangju
    "대구": "대구광역시", # Daegu
    "대전": "대전광역시", # Daejeon
    "부산": "부산광역시", # Busan
    "세종": "세종특별자치시", # Sejong
    "울산": "울산광역시", # Ulsan
    "전남": "전라남도",   # Jeonnam
    "전북": "전라북도",   # Jeonbuk
    "제주": "제주특별자치도", # Jeju
    "충남": "충청남도",   # Chungnam
    "충북": "충청북도",   # Chungbuk
}

CAPITAL_REGION = {"서울특별시", "인천광역시", "경기도"} # Seoul, Incheon, Gyeonggi

In [ ]:
def age_to_group(age: pd.Series) -> pd.Categorical:
    bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, np.inf]
    grouped = pd.cut(age, bins=bins, labels=AGE_GROUPS, right=False, include_lowest=True)
    return pd.Categorical(grouped, categories=AGE_GROUPS, ordered=True)

def normalize_sigungu(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.split().str[0]

def map_place_type(code: pd.Series) -> pd.Series:
    code = pd.to_numeric(code, errors="coerce")
    return np.select(
        [code.eq(1), code.isin([2, 3]), code.eq(97)],
        ["H", "W", "O"],
        default="Unknown",
    )

def assign_time_zone(hour: pd.Series) -> pd.Series:
    hour = pd.to_numeric(hour, errors="coerce").mod(24)
    out = pd.Series(np.nan, index=hour.index, dtype="object")
    out.loc[hour.between(7, 10)] = "T1"
    out.loc[hour.between(11, 14)] = "T2"
    out.loc[hour.between(15, 18)] = "T3"
    out.loc[hour.between(19, 22)] = "T4"
    out.loc[(hour == 23) | hour.between(0, 2)] = "T5"
    out.loc[hour.between(3, 6)] = "T6"
    return out

def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    values = pd.to_numeric(values, errors="coerce")
    weights = pd.to_numeric(weights, errors="coerce")
    mask = values.notna() & weights.notna()
    if not mask.any():
        return np.nan
    return np.average(values[mask], weights=weights[mask])

def build_seoul_code_lookup(zone_id_file: Path) -> pd.DataFrame:
    lookup = pd.read_excel(zone_id_file)
    lookup = lookup.loc[lookup["시도"] == "서울특별시", ["2021_행정동코드(10자리)", "2016_행정구역분류(통계청)"]].copy()
    lookup = lookup.rename(columns={
        "2021_행정동코드(10자리)": "zone_id_2021",
        "2016_행정구역분류(통계청)": "zone_id",
    })
    lookup["zone_id_2021"] = pd.to_numeric(lookup["zone_id_2021"], errors="coerce").astype("Int64")
    lookup["zone_id"] = pd.to_numeric(lookup["zone_id"], errors="coerce").astype("Int64").astype("string")
    lookup = lookup.drop_duplicates(subset=["zone_id_2021"])
    return lookup

## 1. Load and preprocess HTS person records

In [ ]:
def load_hts_person_ids_from_trip(trip_file: Path) -> pd.Series:
    trip = pd.read_csv(trip_file, encoding="cp949", usecols=["idx"])
    trip["person_id"] = pd.to_numeric(trip["idx"], errors="coerce").astype("Int64")
    return trip["person_id"].dropna().drop_duplicates()

# 1. Load and preprocess HTS person records
def load_hts_person(person_file: Path, trip_file: Path, zone_id_file: Path) -> pd.DataFrame:
    # Read person IDs that actually appear in the HTS trip file.
    valid_person_ids = load_hts_person_ids_from_trip(trip_file)

    # Read the respondent-level HTS file.
    # Keep only the fields needed for residence, gender, and age,
    # which are required to construct person-level expansion cells.
    usecols = ["idx", "SQ1_5", "SQ1_6", "SQ1_7", "SQ1_8", "SQ2", "SQ3", "SQ3_1"]
    person = pd.read_csv(person_file, encoding="cp949", usecols=usecols)

    # Rename raw survey variables to descriptive column names.
    person = person.rename(columns={
        "idx":   "person_id",
        "SQ1_5": "residence_zone_id_2021",     # zone_id (2021 year)
        "SQ1_6": "residence_sido",             # province-level
        "SQ1_7": "residence_sigungu",          # city/county/district
        "SQ1_8": "residence_dongeupmyeon",     # administrative district
        "SQ2":   "gender",
        "SQ3":   "birth_year",
        "SQ3_1": "age",
    })

    # Standardize data types and clean string fields.
    # Age is also converted into the common analysis age groups.
    person["person_id"] = pd.to_numeric(person["person_id"], errors="coerce").astype("Int64")
    person["residence_zone_id_2021"] = pd.to_numeric(person["residence_zone_id_2021"], errors="coerce").astype("Int64")
    person["residence_sido"] = person["residence_sido"].astype(str).str.strip().replace(SIDO_MAP)
    person["residence_sigungu"] = person["residence_sigungu"].astype(str).str.strip()
    person["residence_dongeupmyeon"] = person["residence_dongeupmyeon"].astype(str).str.strip()
    person["gender"] = person["gender"].replace(SEX_MAP)
    person["age"] = pd.to_numeric(person["age"], errors="coerce")
    person["age_group"] = age_to_group(person["age"])

    # Keep only respondents who appear in the HTS trip file.
    person = person.loc[person["person_id"].isin(valid_person_ids)].copy()

    # Keep one record per respondent.
    person = person.drop_duplicates(subset=["person_id"]).copy()

    # Convert the 2021 residence zone code to the common zone_id system "the 2016 residence zone code" 
    # used in the Seoul OD analysis.
    seoul_lookup = build_seoul_code_lookup(zone_id_file)
    person = person.merge(
        seoul_lookup.rename(columns={"zone_id_2021": "residence_zone_id_2021", "zone_id": "residence_zone_id"}),
        on="residence_zone_id_2021",
        how="left",
    )

    # Normalize sigungu names (city/county/district level) for Incheon and Gyeonggi
    # so that regional expansion cells are matched consistently.
    person.loc[person["residence_sido"].eq("인천광역시") | person["residence_sido"].eq("경기도"), "residence_sigungu"] = normalize_sigungu(
        person.loc[person["residence_sido"].eq("인천광역시") | person["residence_sido"].eq("경기도"), "residence_sigungu"]
    )

    # Assign the geographic level used for person-level expansion.
    # - Seoul residents: administrative-district level
    # - Incheon/Gyeonggi residents: city/county/district level
    # - Other regions: province level
    # This follows the HTS expansion framework described in the manuscript.
    person["weight_system"] = np.select(
        [
            person["residence_sido"].eq("서울특별시"),
            person["residence_sido"].isin(["인천광역시", "경기도"]),
        ],
        [
            "administrative_district_level",
            "city_county_district_level",
        ],
        default="province_level",
    )

    person["weight_geo_key"] = np.select(
        [
            person["weight_system"].eq("administrative_district_level"),
            person["weight_system"].eq("city_county_district_level"),
        ],
        [
            person["residence_zone_id"],
            person["residence_sido"] + "|" + person["residence_sigungu"],
        ],
        default=person["residence_sido"],
    )

    # Return only the respondent attributes needed for expansion
    # and for linking person-level weights to trip records.
    keep = [
        "person_id",
        "residence_zone_id_2021",  # zone_id (2021 year)
        "residence_zone_id",       # zone_id (2016 year)
        "residence_sido",          # province-level
        "residence_sigungu",       # city/county/district
        "residence_dongeupmyeon",  # administrative district
        "gender",
        "birth_year",
        "age",
        "age_group",
        "weight_system",  # Geographic level used to define the respondent's expansion cell
        "weight_geo_key"  # Geographic identifier used in the person-level weighting cell
    ]
    return person[keep].copy()

hts_person = load_hts_person(HTS_PERSON_FILE, HTS_TRIP_FILE, ZONE_ID_FILE)
print(f"Observed HTS persons: {len(hts_person):,}")

# Inspect key respondent attributes used in the weighting framework.
cols = ["person_id", "residence_zone_id_2021", "residence_sido", "residence_sigungu", "residence_dongeupmyeon", "gender", "age", "age_group", "weight_system"]
hts_person[cols].head()

## 2. Build population tables for weighting

In [ ]:
def build_korea_population_long(pop_korea_file: Path) -> pd.DataFrame:
    # Load the Korea-wide population table.
    # This table is used to construct reference populations
    # for non-Seoul weighting cells.
    pop = pd.read_excel(pop_korea_file)
    pop = pop.rename(columns={"행정기관코드": "zone_id_2021"})

    address = pop["행정기관"].astype(str).str.strip().str.split(r"\s+", n=2, expand=True)
    pop["residence_sido"] = address[0].replace(SIDO_MAP)
    pop["residence_sigungu"] = address[1].fillna(address[0]).astype(str).str.strip()

    # Normalize sigungu names for Incheon and Gyeonggi
    # so that they match the HTS geographic keys consistently.
    pop.loc[pop["residence_sido"].isin(["인천광역시", "경기도"]), "residence_sigungu"] = normalize_sigungu(
        pop.loc[pop["residence_sido"].isin(["인천광역시", "경기도"]), "residence_sigungu"]
    )

    # Map raw male population columns to the common age-group labels.
    male_cols = {
        "남 0~9세": "0-9 years",
        "남 10~19세": "10-19 years",
        "남 20~29세": "20-29 years",
        "남 30~39세": "30-39 years",
        "남 40~49세": "40-49 years",
        "남 50~59세": "50-59 years",
        "남 60~69세": "60-69 years",
        "남 70~79세": "70-79 years",
        "남 80~89세": "80 years and above",
        "남 90~99세": "80 years and above",
        "남 100세 이상": "80 years and above",
    }
    # Map raw female population columns to the common age-group labels.
    female_cols = {
        "여 0~9세": "0-9 years",
        "여 10~19세": "10-19 years",
        "여 20~29세": "20-29 years",
        "여 30~39세": "30-39 years",
        "여 40~49세": "40-49 years",
        "여 50~59세": "50-59 years",
        "여 60~69세": "60-69 years",
        "여 70~79세": "70-79 years",
        "여 80~89세": "80 years and above",
        "여 90~99세": "80 years and above",
        "여 100세 이상": "80 years and above",
    }

    # Convert population values to numeric form.
    pop_value_cols = list(male_cols) + list(female_cols)
    for col in pop_value_cols:
        pop[col] = pd.to_numeric(pop[col].astype(str).str.replace(",", "", regex=False), errors="coerce").fillna(0)

    # Reshape male population columns to long format.
    male_long = pop.melt(
        id_vars=["zone_id_2021", "행정기관", "residence_sido", "residence_sigungu"],
        value_vars=list(male_cols),
        var_name="raw_age_col",
        value_name="population",
    )
    male_long["gender"] = "Male"
    male_long["age_group"] = male_long["raw_age_col"].map(male_cols)

    # Reshape female population columns to long format.
    female_long = pop.melt(
        id_vars=["zone_id_2021", "행정기관", "residence_sido", "residence_sigungu"],
        value_vars=list(female_cols),
        var_name="raw_age_col",
        value_name="population",
    )
    female_long["gender"] = "Female"
    female_long["age_group"] = female_long["raw_age_col"].map(female_cols)

    # Combine male and female population tables,
    # then aggregate to one row per geography × gender × age group.
    pop_long = pd.concat([male_long, female_long], ignore_index=True)
    pop_long = (
        pop_long.groupby(["zone_id_2021", "행정기관", "residence_sido", "residence_sigungu", "gender", "age_group"], as_index=False)["population"]
        .sum()
    )
    # Standardize key types and preserve the common age-group order.
    pop_long["zone_id_2021"] = pd.to_numeric(pop_long["zone_id_2021"], errors="coerce").astype("Int64")
    pop_long["age_group"] = pd.Categorical(pop_long["age_group"], categories=AGE_GROUPS, ordered=True)
    return pop_long

def build_seoul_population_table(pop_seoul_file: Path, zone_id_file: Path) -> pd.DataFrame:
    """
    Build the Seoul population table for HTS expansion.

    Flow:
    1) Load Seoul population by 2021 administrative district.
    2) Harmonize gender and age groups.
    3) Map 2021 zone id to the analysis zone system.
    4) Re-aggregate population by mapped zone × gender × age group.
    """
    # Load the Seoul population table.
    pop = pd.read_excel(pop_seoul_file).copy()

    # Keep the 2021 population only
    pop["통계연도"] = pd.to_numeric(pop["통계연도"], errors="coerce")
    pop = pop.loc[pop["통계연도"].eq(2021)].copy()

    # Harmonize gender
    pop["gender"] = (
        pop["성별명"]
        .astype(str)
        .str.strip()
        .replace({"남자": "Male", "여자": "Female"})
    )

    # Harmonize age groups to the common 10-year categories
    def map_age_group(label: str) -> str:
        label = str(label).strip()
        if "+" in label:
            return "80 years and above"

        nums = re.findall(r"\d+", label)
        if not nums:
            return np.nan

        lower = int(nums[0])
        if lower >= 80:
            return "80 years and above"
        return f"{(lower // 10) * 10}-{(lower // 10) * 10 + 9} years"

    pop["age_group"] = pop["나이대명"].apply(map_age_group)
    pop["age_group"] = pd.Categorical(pop["age_group"], categories=AGE_GROUPS, ordered=True)

    # Population count
    pop["population"] = pd.to_numeric(pop["인구수"], errors="coerce").fillna(0)

    # Aggregate first at the raw 2021 administrative district level
    pop = (
        pop.groupby(["행정동코드", "gender", "age_group"], dropna=False, as_index=False)["population"]
        .sum()
        .rename(columns={"행정동코드": "zone_id_2021"})
    )
    pop["zone_id_2021"] = pd.to_numeric(pop["zone_id_2021"], errors="coerce").astype("Int64")

    # Map 2021 zone id to the analysis zone system
    seoul_lookup = build_seoul_code_lookup(zone_id_file)
    pop = pop.merge(seoul_lookup, on="zone_id_2021", how="left")
    pop = pop.rename(columns={"zone_id": "residence_zone_id"})

    # Diagnostic: check unmatched Seoul zone id
    unmatched = pop["residence_zone_id"].isna().sum()
    if unmatched > 0:
        print(f"Warning: {unmatched:,} Seoul population rows could not be mapped to analysis zones.")
        pop = pop.loc[pop["residence_zone_id"].notna()].copy()

    # Critical step:
    # re-aggregate AFTER mapping, because multiple 2021 zone id may map to one analysis zone
    pop = (
        pop.groupby(["residence_zone_id", "gender", "age_group"], dropna=False, as_index=False)["population"]
        .sum()
    )

    return pop[["residence_zone_id", "gender", "age_group", "population"]].copy()

pop_korea = build_korea_population_long(POP_KOREA_FILE)
pop_seoul = build_seoul_population_table(POP_SEOUL_FILE, ZONE_ID_FILE)

# Inspect the final Seoul population table used for expansion.
pop_seoul

## 3. Derive person-level expansion coefficients

In [ ]:
def compute_seoul_weight_table(person: pd.DataFrame, pop_seoul: pd.DataFrame) -> pd.DataFrame:
    """
    Compute Seoul person-level expansion factors
    using mapped analysis zone × gender × age group cells.
    """
    # Count HTS respondents within each Seoul weighting cell.
    # Seoul weights are defined at the administrative-district level
    # after mapping residence codes to the common analysis zone system.
    hts_cell = (
        person.loc[person["weight_system"].eq("administrative_district_level")]
        .groupby(["residence_zone_id", "gender", "age_group"], dropna=False)["person_id"]
        .nunique()
        .reset_index(name="n_hts")
    )

    # Merge the Seoul reference population with the HTS respondent counts
    # for the same zone × gender × age-group cells.
    weight_table = pop_seoul.merge(
        hts_cell,
        on=["residence_zone_id", "gender", "age_group"],
        how="left"
    )

    # Set missing HTS counts to zero and compute the person-level expansion factor.
    # Cells with no HTS respondents are left as missing because no direct weight can be assigned from the observed sample.
    weight_table["n_hts"] = weight_table["n_hts"].fillna(0).astype(int)
    weight_table["person_weight"] = np.where(
        weight_table["n_hts"] > 0,
        weight_table["population"] / weight_table["n_hts"],
        np.nan
    )

    # Store the weighting-system label and geographic key
    # so this table can later be combined with the non-Seoul weights.
    weight_table["weight_system"] = "administrative_district_level"
    weight_table["weight_geo_key"] = weight_table["residence_zone_id"]

    # Return the final Seoul weight table.
    return weight_table[
        ["weight_system", "weight_geo_key", "gender", "age_group", "population", "n_hts", "person_weight"]
    ].copy()

def compute_nonseoul_weight_table(person: pd.DataFrame, pop_korea: pd.DataFrame) -> pd.DataFrame:
    # Exclude Seoul because Seoul weights are handled separately
    # at the administrative-district level.
    pop_nonseoul = pop_korea.loc[~pop_korea["residence_sido"].eq("서울특별시")].copy()
    # Assign the weighting system for non-Seoul residents.
    # - Incheon/Gyeonggi: city/county/district level
    # - Other regions: province level
    pop_nonseoul["weight_system"] = np.select(
        [
            pop_nonseoul["residence_sido"].isin(["인천광역시", "경기도"]),
        ],
        [
            "city_county_district_level",
        ],
        default="province_level",
    )
    # Construct the geographic key used to define each non-Seoul weighting cell.
    pop_nonseoul["weight_geo_key"] = np.where(
        pop_nonseoul["weight_system"].eq("city_county_district_level"),
        pop_nonseoul["residence_sido"] + "|" + pop_nonseoul["residence_sigungu"],
        pop_nonseoul["residence_sido"],
    )

    # Aggregate the reference population to the non-Seoul weighting cells.
    pop_cell = (
        pop_nonseoul.groupby(["weight_system", "weight_geo_key", "gender", "age_group"], dropna=False)["population"]
        .sum()
        .reset_index()
    )

    # Count HTS respondents within the same non-Seoul weighting cells.
    hts_cell = (
        person.loc[~person["weight_system"].eq("administrative_district_level")]
        .groupby(["weight_system", "weight_geo_key", "gender", "age_group"], dropna=False)["person_id"]
        .nunique()
        .reset_index(name="n_hts")
    )

    # Merge the reference population and HTS respondent counts,
    # then compute the person-level expansion factor.
    weight_table = pop_cell.merge(hts_cell, on=["weight_system", "weight_geo_key", "gender", "age_group"], how="left")
    weight_table["n_hts"] = weight_table["n_hts"].fillna(0).astype(int)
    weight_table["person_weight"] = np.where(weight_table["n_hts"] > 0, weight_table["population"] / weight_table["n_hts"], np.nan)
    return weight_table

def attach_person_weights(person: pd.DataFrame, seoul_weights: pd.DataFrame, nonseoul_weights: pd.DataFrame) -> pd.DataFrame:
    # Combine Seoul and non-Seoul weight tables
    # into one person-level weighting reference.
    weight_table = pd.concat([seoul_weights, nonseoul_weights], ignore_index=True)
    # Attach the expansion factor to each HTS respondent
    # using the respondent's weighting cell.
    weighted = person.merge(
        weight_table[["weight_system", "weight_geo_key", "gender", "age_group", "person_weight"]],
        on=["weight_system", "weight_geo_key", "gender", "age_group"],
        how="left",
    )
    # Record which weighting system was used for each respondent.
    weighted["weight_source"] = weighted["weight_system"]
    return weighted

seoul_weights = compute_seoul_weight_table(hts_person, pop_seoul)
nonseoul_weights = compute_nonseoul_weight_table(hts_person, pop_korea)
hts_person_weighted = attach_person_weights(hts_person, seoul_weights, nonseoul_weights)

print(f"Weighted HTS persons: {hts_person_weighted['person_weight'].sum():,.0f}")
print(f"Missing person weights: {hts_person_weighted['person_weight'].isna().sum():,}")

# Inspect the final person-level HTS table with attached expansion factors.
cols = ["person_id", "residence_zone_id_2021", "residence_sido", "residence_sigungu", "residence_dongeupmyeon", "gender", "age_group", "person_weight"]
HTS_PERSON_WEIGHTED = hts_person_weighted[cols].copy()
HTS_PERSON_WEIGHTED.head()

In [ ]:
# Save
HTS_PERSON_WEIGHTED.to_csv(HTS_PERSON_WEIGHTED_FILE, index=False, encoding="cp949")

print('Save complete')
# print(HTS_PERSON_WEIGHTED_FILE)

### (Additional Analysis) Summary of the Weighting Structure for the Full HTS Person Sample

In [ ]:
overall_summary = pd.DataFrame({
    "Observed_HTS_persons": [HTS_PERSON_WEIGHTED["person_id"].nunique()],
    "Weighted_population_equivalent": [HTS_PERSON_WEIGHTED["person_weight"].sum()],
    "Mean_weight": [HTS_PERSON_WEIGHTED["person_weight"].mean()],
    "Min_weight": [HTS_PERSON_WEIGHTED["person_weight"].min()],
    "Median weight": [HTS_PERSON_WEIGHTED["person_weight"].median()],
    "Max_weight": [HTS_PERSON_WEIGHTED["person_weight"].max()],
})
overall_summary["Mean_expansion_factor"] = (
    overall_summary["Weighted_population_equivalent"] / overall_summary["Observed_HTS_persons"]
)
overall_summary["Observed_to_weighted_ratio_(%)"] = (
    overall_summary["Observed_HTS_persons"] / overall_summary["Weighted_population_equivalent"] * 100
)

# ---------------------------------------------------------
# [0] OVERALL
# ---------------------------------------------------------
print("=== Overall person-level weighting summary (HTS) ===")
print(overall_summary)

In [ ]:
def make_weight_summary(df, group_cols):
    """
    Calculation by sub-group: observed persons, weighted persons, mean expansion factor,
    observed to weighted ratio
    """
    out = (
        df.groupby(group_cols, dropna=False)
          .agg(
              observed_HTS_persons=("person_id", "nunique"),
              weighted_persons=("person_weight", "sum"),
              mean_weight=("person_weight", "mean"),
              min_weight=("person_weight", "min"),
              median_weight=("person_weight", "median"),
              max_weight=("person_weight", "max"),
          )
          .reset_index()
    )
    out["mean_expansion_factor"] = out["weighted_persons"] / out["observed_HTS_persons"]
    out["observed_to_weighted_ratio_(%)"] = (
        out["observed_HTS_persons"] / out["weighted_persons"] * 100
    )
    return out


# English mapping
SIDO_MAP_EN = {
    "서울특별시": "Seoul",
    "인천광역시": "Incheon",
    "경기도": "Gyeonggi",
    "강원도": "Gangwon",
    "경상남도": "Gyeongnam",
    "경상북도": "Gyeongbuk",
    "광주광역시": "Gwangju",
    "대구광역시": "Daegu",
    "대전광역시": "Daejeon",
    "부산광역시": "Busan",
    "세종특별자치시": "Sejong",
    "울산광역시": "Ulsan",
    "전라남도": "Jeonnam",
    "전라북도": "Jeonbuk",
    "제주특별자치도": "Jeju",
    "충청남도": "Chungnam",
    "충청북도": "Chungbuk",
}
# Apply mapping
HTS_PERSON_WEIGHTED["Residence_region"] = (
    HTS_PERSON_WEIGHTED["residence_sido"]
    .map(SIDO_MAP_EN)
)

# ---------------------------------------------------------
# [1] Regional Expansion Result Structure Table
# ---------------------------------------------------------
table_region = make_weight_summary(
    HTS_PERSON_WEIGHTED,
    group_cols=["Residence_region"]
).sort_values("weighted_persons", ascending=False)

print("\n=== Table: Weighting structure by residence region ===")
print(table_region.head(50))

# ---------------------------------------------------------
# [2] Gender × Age Group Extended Result Structure Table
# ---------------------------------------------------------
table_gender_age = make_weight_summary(
    HTS_PERSON_WEIGHTED,
    group_cols=["gender", "age_group"]
).sort_values(["gender", "age_group"])

print("\n=== Table: Weighting structure by gender and age group ===")
print(table_gender_age)

In [ ]:
# ---------------------------------------------------------
# [3] Gender × Age Group Extended Result Structure Table (Seoul residents)
# ---------------------------------------------------------
HTS_PERSON_WEIGHTED_Seoul = HTS_PERSON_WEIGHTED[HTS_PERSON_WEIGHTED["Residence_region"] == "Seoul"].copy()

table_seoul_gender_age = make_weight_summary(
    HTS_PERSON_WEIGHTED_Seoul,
    group_cols=["gender", "age_group"]
).sort_values(["gender", "age_group"])

print("\n=== Table: Seoul residents only, by gender and age group ===")
print(table_seoul_gender_age)

## 4. Merge person weights into HTS trip records

In [ ]:
def load_hts_trip(trip_file: Path) -> pd.DataFrame:
    # Load the trip-level HTS file.
    # Keep only fields required for OD construction and temporal attributes.
    usecols = [
        "idx", "fid", "th_seq",
        "sTP1_1_6", "sTP1_1_5",
        "TP1_1_6", "TP1_1_5",
        "sTP1", "TP1",
        "TP3_1", "TP3_2", "TP4_1", "TP4_2",
    ]
    trip = pd.read_csv(trip_file, encoding="cp949", usecols=usecols)

    # Rename variables to descriptive names used in the analysis.
    trip = trip.rename(columns={
        "idx":      "person_id",
        "fid":      "trip_id",     # trip identifier
        "th_seq":   "trip_seq",    # trip sequence number
        "sTP1_1_6": "origin_sido",
        "sTP1_1_5": "origin_zone_id_2021",
        "TP1_1_6":  "destination_sido",
        "TP1_1_5":  "destination_zone_id_2021",
        "sTP1":     "origin_place_type_raw",
        "TP1":      "destination_place_type_raw",
        "TP3_1":    "departure_hour",
        "TP3_2":    "departure_minute",
        "TP4_1":    "arrival_hour",
        "TP4_2":    "arrival_minute",
    })

    # Standardize key data types.
    trip["person_id"] = pd.to_numeric(trip["person_id"], errors="coerce").astype("Int64")
    trip["trip_id"] = pd.to_numeric(trip["trip_id"], errors="coerce").astype("Int64")
    trip["trip_seq"] = pd.to_numeric(trip["trip_seq"], errors="coerce")
    trip["origin_sido"] = trip["origin_sido"].astype(str).str.strip()
    trip["destination_sido"] = trip["destination_sido"].astype(str).str.strip()
    trip["origin_zone_id_2021"] = pd.to_numeric(trip["origin_zone_id_2021"], errors="coerce").astype("Int64")
    trip["destination_zone_id_2021"] = pd.to_numeric(trip["destination_zone_id_2021"], errors="coerce").astype("Int64")
    return trip

# Merge person-level expansion factors into trip records,
# retain intra-Seoul trips only, and derive trip attributes
# aligned with the manuscript definition:
# travel type, arrival hour, time zone, and trip-level travel time.
def preprocess_weighted_hts_trips(trip: pd.DataFrame, person_weighted: pd.DataFrame, zone_id_file: Path) -> pd.DataFrame:
    trip = trip.merge(
        person_weighted[["person_id", "gender", "age_group", "person_weight", "weight_source", "residence_sido", "residence_sigungu"]],
        on="person_id",
        how="inner",
    )

    trip = trip.loc[
        trip["origin_sido"].eq("서울특별시")
        & trip["destination_sido"].eq("서울특별시")
        & trip["trip_seq"].gt(0)
    ].copy()

    assert trip["trip_seq"].gt(0).all(), (
        "Non-trip placeholder records remain in the analytical trip table."
    )

    lookup = build_seoul_code_lookup(zone_id_file)
    trip = trip.merge(
        lookup.rename(columns={"zone_id_2021": "origin_zone_id_2021", "zone_id": "origin_id"}),
        on="origin_zone_id_2021",
        how="left",
    )
    trip = trip.merge(
        lookup.rename(columns={"zone_id_2021": "destination_zone_id_2021", "zone_id": "destination_id"}),
        on="destination_zone_id_2021",
        how="left",
    )

    # Derive travel type from origin and destination place types
    trip["origin_place_type"] = map_place_type(trip["origin_place_type_raw"])
    trip["destination_place_type"] = map_place_type(trip["destination_place_type_raw"])
    trip["travel_type"] = trip["origin_place_type"] + trip["destination_place_type"]

    trip["departure_hour"] = pd.to_numeric(trip["departure_hour"], errors="coerce")
    trip["departure_minute"] = pd.to_numeric(trip["departure_minute"], errors="coerce")
    trip["arrival_hour"] = pd.to_numeric(trip["arrival_hour"], errors="coerce")
    trip["arrival_minute"] = pd.to_numeric(trip["arrival_minute"], errors="coerce")

    # Use arrival time as the temporal reference and compute trip-level travel time
    departure_total_min = trip["departure_hour"] * 60 + trip["departure_minute"]
    arrival_total_min = trip["arrival_hour"] * 60 + trip["arrival_minute"]
    travel_time = arrival_total_min - departure_total_min
    trip["travel_time_min"] = np.where(travel_time < 0, travel_time + 24 * 60, travel_time)

    trip["time_zone"] = assign_time_zone(trip["arrival_hour"])
    trip["origin_id"] = trip["origin_id"].astype("string")
    trip["destination_id"] = trip["destination_id"].astype("string")

    # Expanded trips are represented by the person-level expansion factor
    trip["trips"] = trip["person_weight"]

    keep = [
        "person_id",
        "trip_id",
        "trip_seq",
        "gender",
        "age_group",
        "person_weight",
        "weight_source",
        "residence_sido",
        "residence_sigungu",
        "origin_id",
        "destination_id",
        "travel_type",
        "arrival_hour",
        "time_zone",
        "travel_time_min",
        "trips",
    ]
    return trip[keep].copy()

hts_trip_raw = load_hts_trip(HTS_TRIP_FILE)
hts_trip_weighted = preprocess_weighted_hts_trips(hts_trip_raw, hts_person_weighted, ZONE_ID_FILE)

cols = ["person_id", "trip_id", "trip_seq", 
        "gender", "age_group", "person_weight", 
        "origin_id", "destination_id", "travel_type", "arrival_hour", "time_zone", "travel_time_min", "trips"]
HTS_TRIP_WEIGHTED = hts_trip_weighted[cols].copy()
HTS_TRIP_WEIGHTED

In [ ]:
# Save
HTS_TRIP_WEIGHTED.to_csv(HTS_TRIP_WEIGHTED_FILE, index=False, encoding="cp949")

print('Save complete')
# print(HTS_TRIP_WEIGHTED_FILE)

### (Additional Analysis) Summary statistics

In [ ]:
print(f"Raw intra-Seoul HTS trips: {len(hts_trip_weighted):,}")
print(f"Expanded intra-Seoul HTS trips: {hts_trip_weighted['trips'].sum():,.0f}")

In [ ]:
move_nonzero_person = set(hts_trip_raw.loc[hts_trip_raw['trip_seq'] != 0, 'person_id'].dropna().unique())
print("People who have passed through at least once in HTS raw:", f"{len(move_nonzero_person):,}")

move_zero_person = set(hts_trip_raw.loc[hts_trip_raw['trip_seq'] == 0, 'person_id'].dropna().unique())
print("People who have never passed through HTS raw:", f"{len(move_zero_person):,}")

In [ ]:
# Additional summary statistics after counting trip_seq == 0 / != 0 persons

# 1) Person-level table for intra-Seoul travelers
seoul_internal_persons = (
    hts_trip_weighted[["person_id", "person_weight", "residence_sido"]]
    .drop_duplicates(subset=["person_id"])
    .copy()
)

seoul_internal_persons["residence_sido_eng"] = (
    seoul_internal_persons["residence_sido"]
    .astype(str)
    .str.strip()
    .replace(SIDO_MAP_EN)
)

# 2) Intra-Seoul travelers: observed and expanded HTS persons
observed_hts_person_n = seoul_internal_persons["person_id"].nunique()
expanded_hts_person_n = seoul_internal_persons["person_weight"].sum()

print(f"Observed HTS persons (intra-Seoul travelers): {observed_hts_person_n:,}")
print(f"Sum of person-level expansion weights for intra-Seoul travelers: {expanded_hts_person_n:,.0f}")

# 3) Among intra-Seoul travelers (observed persons): Seoul residents
seoul_residents_internal = seoul_internal_persons[
    seoul_internal_persons["residence_sido_eng"].eq("Seoul")
].copy()

print(f"HTS Seoul residents (observed, among intra-Seoul travelers): {seoul_residents_internal['person_id'].nunique():,}")

# 4) Residence-sido distribution among intra-Seoul travelers (observed persons)
res_sido_dist = (
    seoul_internal_persons["residence_sido_eng"]
    .value_counts(dropna=False)
    .rename_axis("Residence Sido")
    .reset_index(name="count")
)

res_sido_dist["percentage"] = (
    res_sido_dist["count"] / res_sido_dist["count"].sum() * 100
).round(2)

print("\nHTS respondent residence-sido distribution among intra-Seoul travelers (observed persons):")
print(res_sido_dist.to_string(index=False))

# 5) Share of non-travelers among Seoul residents appearing in the trip file
seoul_person_ids_in_trip = set(
    hts_trip_raw.loc[
        hts_trip_raw["person_id"].isin(
            seoul_internal_persons.loc[
                seoul_internal_persons["residence_sido_eng"].eq("Seoul"),
                "person_id"
            ]
        ),
        "person_id"
    ].dropna().unique()
)

raw_trip_seoul = hts_trip_raw.loc[
    hts_trip_raw["person_id"].isin(seoul_person_ids_in_trip),
    ["person_id", "trip_seq"]
].copy()

total_seoul_person_n_in_trip = len(seoul_person_ids_in_trip)

print(f"\nSeoul residents appearing in trip records: {total_seoul_person_n_in_trip:,}")

In [ ]:
# ---------------------------------------------------------
# Share of non-travelers among all Seoul residents
# appearing in the HTS trip file
# ---------------------------------------------------------

# 1) Create person-level trip status from the raw HTS trip file
seoul_resident_trip_status = (
    hts_trip_raw.groupby("person_id", dropna=False)
    .agg(
        has_trip=("trip_seq", lambda x: x.ne(0).any()),
        zero_only=("trip_seq", lambda x: x.eq(0).all()),
    )
    .reset_index()
)

# 2) Attach residence information from the weighted person table
seoul_resident_trip_status = seoul_resident_trip_status.merge(
    HTS_PERSON_WEIGHTED[["person_id", "residence_sido"]].drop_duplicates(subset=["person_id"]),
    on="person_id",
    how="left",
)

# 3) Keep only Seoul residents
seoul_resident_trip_status = seoul_resident_trip_status.loc[
    seoul_resident_trip_status["residence_sido"].astype(str).str.strip().eq("서울특별시")
].copy()

# 4) Count total Seoul residents and non-travelers
total_seoul_residents = seoul_resident_trip_status["person_id"].nunique()
seoul_nontravelers = seoul_resident_trip_status.loc[
    seoul_resident_trip_status["zero_only"], "person_id"
].nunique()

seoul_nontraveler_ratio = seoul_nontravelers / total_seoul_residents * 100

# 5) Print results in English
print("[Among all Seoul residents appearing in the HTS trip file]")
print(f"Total Seoul residents: {total_seoul_residents:,}")
print(f"Non-travelers (persons with only trip_seq == 0): {seoul_nontravelers:,}")
print(f"Non-traveler ratio: {seoul_nontraveler_ratio:.2f}%")

### (Additional Analysis) Summary of the Weighting Structure for the IntraSeoul HTS trip sample

In [ ]:
# ---------------------------------------------------------
# Overall intra-Seoul trip summary
# ---------------------------------------------------------

trip_overall = pd.DataFrame({
    "observed_trips": [len(HTS_TRIP_WEIGHTED)],
    "weighted_trips": [HTS_TRIP_WEIGHTED["trips"].sum()],
    "unique_persons_in_trip_sample": [
        HTS_TRIP_WEIGHTED["person_id"].nunique()
    ],
})

print("\n=== Intra-Seoul trip analysis sample summary ===")
print(trip_overall)


# ---------------------------------------------------------
# Person-level summary for respondents with intra-Seoul trips
# ---------------------------------------------------------

intra_seoul_persons = (
    HTS_TRIP_WEIGHTED[
        [
            "person_id",
            "gender",
            "age_group",
            "person_weight",
        ]
    ]
    .drop_duplicates(subset=["person_id"])
    .copy()
)

table_intra_seoul_person_gender_age = make_weight_summary(
    intra_seoul_persons,
    group_cols=["gender", "age_group"],
).sort_values(["gender", "age_group"])

print(
    "\n=== Persons with at least one intra-Seoul trip, "
    "by gender and age group ==="
)
print(table_intra_seoul_person_gender_age)

In [ ]:
# ---------------------------------------------------------
# [6] Trip Distribution (Intra Seoul trip)
# ---------------------------------------------------------
def make_trip_distribution(df, group_cols):
    out = (
        df.groupby(group_cols, dropna=False)
          .agg(
              observed_trips=("trip_id", "count"),
              weighted_trips=("trips", "sum")
          )
          .reset_index()
    )
    out["observed_trip_share_pct"] = out["observed_trips"] / out["observed_trips"].sum() * 100
    out["weighted_trip_share_pct"] = out["weighted_trips"] / out["weighted_trips"].sum() * 100
    return out

table_trip_gender_age = make_trip_distribution(
    HTS_TRIP_WEIGHTED,
    group_cols=["gender", "age_group"]
).sort_values(["gender", "age_group"])

print("\n=== Intra-Seoul trip-level distribution by gender and age group ===")
print(table_trip_gender_age)

In [ ]:
output_file = (
    Additional_RESULT_DIR
    / "Summary_of_the_Weighting_Structure.xlsx"
)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    overall_summary.to_excel(
        writer,
        sheet_name="Overall_Full",
        index=False,
    )
    trip_overall.to_excel(
        writer,
        sheet_name="Overall_IntraSeoul",
        index=False,
    )
    table_region.to_excel(
        writer,
        sheet_name="Region_Full",
        index=False,
    )
    table_gender_age.to_excel(
        writer,
        sheet_name="GenderAge_Full",
        index=False,
    )
    table_seoul_gender_age.to_excel(
        writer,
        sheet_name="GenderAge_Seoul",
        index=False,
    )
    table_intra_seoul_person_gender_age.to_excel(
        writer,
        sheet_name="GenderAge_IntraPerson",
        index=False,
    )
    table_trip_gender_age.to_excel(
        writer,
        sheet_name="GenderAge_IntraTrip",
        index=False,
    )

print("completed 'Summary_of_the_Weighting_Structure.xlsx'")
# print(f"\nSaved to: {output_file}")

## 5. Convert weighted HTS trips to the common HTS-LCS format

In [ ]:
# This step creates a full 424 × 424 Seoul OD frame using the zone-id file.
# Weighted HTS trips are then aggregated onto that OD space.
# To avoid creating an excessively large fully crossed table,
# each subgroup table is built separately.

from pathlib import Path
import itertools
import numpy as np
import pandas as pd


def standardize_zone_id(series: pd.Series) -> pd.Series:
    """
    Convert zone IDs to a consistent string format.
    This avoids merge errors caused by mixed int/string types.
    """
    return (
        pd.to_numeric(series, errors="coerce")
        .astype(np.int64)
        .astype(str)
    )


def prepare_hts_trip_keys(hts_trip_weighted: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize OD key columns in the weighted HTS trip table.
    """
    df = hts_trip_weighted.copy()

    df["origin_id"] = standardize_zone_id(df["origin_id"])
    df["destination_id"] = standardize_zone_id(df["destination_id"])

    return df


def build_od_frame(zone_id_file: Path) -> pd.DataFrame:
    """
    Build the complete Seoul OD frame (424 × 424) from the zone-id file.
    The OD keys are stored as strings for merge stability.
    """
    zone_lookup = pd.read_excel(zone_id_file)

    zone_lookup = zone_lookup.loc[
        zone_lookup["시도"].eq("서울특별시"),
        ["2016_행정구역분류(통계청)"],
    ].copy()

    zone_ids = (
        pd.to_numeric(zone_lookup["2016_행정구역분류(통계청)"], errors="coerce")
        .dropna()
        .astype(np.int64)
        .astype(str)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    od_pairs = list(itertools.product(zone_ids, repeat=2))
    od_df = pd.DataFrame(od_pairs, columns=["origin_id", "destination_id"])

    od_df["OD_pair"] = od_df["origin_id"] + "_" + od_df["destination_id"]

    return od_df[["OD_pair", "origin_id", "destination_id"]].copy()


def aggregate_od_trips(
    hts_trip_weighted: pd.DataFrame,
    od_frame: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aggregate weighted HTS trips to the OD level only.
    Unobserved OD cells are filled with zero.
    """
    df = prepare_hts_trip_keys(hts_trip_weighted)

    observed = (
        df.groupby(["origin_id", "destination_id"], dropna=False, as_index=False)["trips"]
        .sum()
    )

    table = od_frame.merge(
        observed,
        on=["origin_id", "destination_id"],
        how="left",
    )

    table["trips"] = table["trips"].fillna(0)

    table = table[
        ["OD_pair", "origin_id", "destination_id", "trips"]
    ].sort_values(["origin_id", "destination_id"]).reset_index(drop=True)

    return table


def build_attribute_levels(
    hts_trip_weighted: pd.DataFrame,
    attr_cols: list[str],
) -> pd.DataFrame:
    """
    Extract unique attribute combinations observed in the HTS trip table.
    """
    df = hts_trip_weighted.copy()

    levels = (
        df[attr_cols]
        .drop_duplicates()
        .sort_values(attr_cols)
        .reset_index(drop=True)
    )

    return levels


def build_od_attribute_table(
    hts_trip_weighted: pd.DataFrame,
    od_frame: pd.DataFrame,
    attr_col: str,
) -> pd.DataFrame:
    """
    Build an OD × single-attribute weighted trip table.
    The table contains all OD pairs crossed with all observed levels
    of the given attribute, with missing cells filled with zero.
    """
    df = prepare_hts_trip_keys(hts_trip_weighted)

    observed = (
        df.groupby(
            ["origin_id", "destination_id", attr_col],
            dropna=False,
            as_index=False,
        )["trips"]
        .sum()
    )

    attr_levels = build_attribute_levels(df, [attr_col])

    # Cross join: OD × attribute levels
    full_frame = od_frame.merge(attr_levels, how="cross")

    table = full_frame.merge(
        observed,
        on=["origin_id", "destination_id", attr_col],
        how="left",
    )

    table["trips"] = table["trips"].fillna(0)

    table = table[
        ["OD_pair", "origin_id", "destination_id", attr_col, "trips"]
    ].sort_values(["origin_id", "destination_id", attr_col]).reset_index(drop=True)

    return table


def build_od_multi_attribute_table(
    hts_trip_weighted: pd.DataFrame,
    od_frame: pd.DataFrame,
    attr_cols: list[str],
) -> pd.DataFrame:
    """
    Build an OD × multi-attribute weighted trip table.
    The table contains all OD pairs crossed with all observed attribute
    combinations, with missing cells filled with zero.
    """
    df = prepare_hts_trip_keys(hts_trip_weighted)

    observed = (
        df.groupby(
            ["origin_id", "destination_id"] + attr_cols,
            dropna=False,
            as_index=False,
        )["trips"]
        .sum()
    )

    attr_levels = build_attribute_levels(df, attr_cols)

    # Cross join: OD × attribute-combination levels
    full_frame = od_frame.merge(attr_levels, how="cross")

    table = full_frame.merge(
        observed,
        on=["origin_id", "destination_id"] + attr_cols,
        how="left",
    )

    table["trips"] = table["trips"].fillna(0)

    table = table[
        ["OD_pair", "origin_id", "destination_id"] + attr_cols + ["trips"]
    ].sort_values(["origin_id", "destination_id"] + attr_cols).reset_index(drop=True)

    return table

In [ ]:
OD_FRAME = build_od_frame(ZONE_ID_FILE)

print(f"Number of Seoul zones: {OD_FRAME['origin_id'].nunique():,}")
print(f"Number of OD pairs: {len(OD_FRAME):,}")

OD_HTS = aggregate_od_trips(
    hts_trip_weighted=HTS_TRIP_WEIGHTED,
    od_frame=OD_FRAME,
)

OD_HTS_gender = build_od_attribute_table(
    hts_trip_weighted=HTS_TRIP_WEIGHTED,
    od_frame=OD_FRAME,
    attr_col="gender",
)

OD_HTS_agegroup = build_od_attribute_table(
    hts_trip_weighted=HTS_TRIP_WEIGHTED,
    od_frame=OD_FRAME,
    attr_col="age_group",
)

OD_HTS_traveltype = build_od_attribute_table(
    hts_trip_weighted=HTS_TRIP_WEIGHTED,
    od_frame=OD_FRAME,
    attr_col="travel_type",
)

OD_HTS_timezone = build_od_attribute_table(
    hts_trip_weighted=HTS_TRIP_WEIGHTED,
    od_frame=OD_FRAME,
    attr_col="time_zone",
)

OD_HTS_traveltype_timezone = build_od_multi_attribute_table(
    hts_trip_weighted=HTS_TRIP_WEIGHTED,
    od_frame=OD_FRAME,
    attr_cols=["travel_type", "time_zone"],
)

print("OD matrix creation by subgroup completed")

In [ ]:
OD_HTS_gender

In [ ]:
OD_HTS_agegroup

In [ ]:
OD_HTS_traveltype

In [ ]:
OD_HTS_timezone

In [ ]:
OD_HTS_traveltype_timezone.head(20)

In [ ]:
# Save outputs
OD_HTS.to_csv(OD_HTS_FILE, index=False, encoding="utf-8-sig")
OD_HTS_gender.to_csv(OD_HTS_GENDER_FILE, index=False, encoding="utf-8-sig")
OD_HTS_agegroup.to_csv(OD_HTS_AGEGROUP_FILE, index=False, encoding="utf-8-sig")
OD_HTS_traveltype.to_csv(OD_HTS_TRAVELTYPE_FILE, index=False, encoding="utf-8-sig")
OD_HTS_timezone.to_csv(OD_HTS_TIMEZONE_FILE, index=False, encoding="utf-8-sig")
OD_HTS_traveltype_timezone.to_csv(OD_HTS_TRAVELTYPE_TIMEZONE_FILE, index=False, encoding="utf-8-sig")

print("OD matrix creation by subgroup completed")
# print(f"Saved: {OD_HTS_FILE}")
# print(f"Saved: {OD_HTS_GENDER_FILE}")
# print(f"Saved: {OD_HTS_AGEGROUP_FILE}")
# print(f"Saved: {OD_HTS_TRAVELTYPE_FILE}")
# print(f"Saved: {OD_HTS_TIMEZONE_FILE}")
# print(f"Saved: {OD_HTS_TRAVELTYPE_TIMEZONE_FILE}")

### (Additional Analysis) Subgroup-Level Trip Distribution (Percent and Count)

In [ ]:
# Summarize raw and expanded intra-Seoul HTS trip distributions
# by gender, age group, travel type, and time zone.

def summarize_raw_expanded_distribution(
    df: pd.DataFrame,
    group_col: str,
) -> pd.DataFrame:
    """
    Create distribution tables for:
    - Raw intra-Seoul HTS trips: row counts
    - Expanded intra-Seoul HTS trips: sum of 'trips'

    Parameters
    ----------
    df : pd.DataFrame
        Weighted HTS trip table.
    group_col : str
        Column used for grouping.

    Returns
    -------
    pd.DataFrame
        Summary table with raw and expanded counts and percentages.
    """
    raw_total = len(df)
    expanded_total = df["trips"].sum()

    summary = (
        df.groupby(group_col, dropna=False)
        .agg(
            raw_count=("trip_id", "size"),
            expanded_count=("trips", "sum"),
        )
        .reset_index()
    )

    summary["raw_pct"] = summary["raw_count"] / raw_total * 100
    summary["expanded_pct"] = summary["expanded_count"] / expanded_total * 100

    summary = summary.sort_values(group_col).reset_index(drop=True)

    return summary


# Create summary tables
HTS_DIST_GENDER = summarize_raw_expanded_distribution(hts_trip_weighted, "gender")
HTS_DIST_AGEGROUP = summarize_raw_expanded_distribution(hts_trip_weighted, "age_group")
HTS_DIST_TRAVELTYPE = summarize_raw_expanded_distribution(hts_trip_weighted, "travel_type")
HTS_DIST_TIMEZONE = summarize_raw_expanded_distribution(hts_trip_weighted, "time_zone")


# Optional: format percentages for easier reading
def format_distribution_table(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    out = df.copy()
    out["raw_count"] = out["raw_count"].astype(int)
    out["expanded_count"] = out["expanded_count"].round(0).astype(int)
    out["raw_pct"] = out["raw_pct"].round(1)
    out["expanded_pct"] = out["expanded_pct"].round(1)
    out = out.rename(columns={
        group_col: "category",
        "raw_count": "Raw HTS trips (n)",
        "raw_pct": "Raw HTS trips (%)",
        "expanded_count": "Expanded HTS trips (n)",
        "expanded_pct": "Expanded HTS trips (%)",
    })
    return out


HTS_DIST_GENDER_FMT = format_distribution_table(HTS_DIST_GENDER, "gender")
HTS_DIST_AGEGROUP_FMT = format_distribution_table(HTS_DIST_AGEGROUP, "age_group")
HTS_DIST_TRAVELTYPE_FMT = format_distribution_table(HTS_DIST_TRAVELTYPE, "travel_type")
HTS_DIST_TIMEZONE_FMT = format_distribution_table(HTS_DIST_TIMEZONE, "time_zone")


# Print results
print("\n[Gender distribution]")
display(HTS_DIST_GENDER_FMT)

print("\n[Age-group distribution]")
display(HTS_DIST_AGEGROUP_FMT)

print("\n[Travel-type distribution]")
display(HTS_DIST_TRAVELTYPE_FMT)

print("\n[Time-zone distribution]")
display(HTS_DIST_TIMEZONE_FMT)

# Notes

- Expansion coefficients are derived at the **person level** using the full HTS respondent file.
- The resulting person weights are then applied to all trips reported by the same respondent.
- The dataset is restricted to **intra-Seoul trips only after person-level weighting**.
- The final `hts_common` table is aligned to the LCS structure through common OD identifiers, gender, age group, travel type, arrival hour, and time zone.

## Expected outputs

Using the data version analyzed in the paper, the HTS preprocessing
notebook should produce approximately:

- 113,763 observed HTS persons
- 50,890,662 weighted population equivalent
- 57,007 unweighted intra-Seoul trips
- 20,764,048 expanded intra-Seoul trips
- 19,611 respondents with at least one intra-Seoul trip
- 424 Seoul analysis zones
- 179,776 possible OD pairs